# 01 - Data Loading & Cleaning
**Objective:** Load raw datasets from World Bank and WTO, clean inconsistencies, handle missing values, and save standardized outputs.
**Data Sources:**
- World Bank Open Data – GDP, GDP growth, GDP per capita, population, trade (% of GDP), imports (% of GDP)
- WTO Statistics – Merchandise trade values by product group for Algeria
**Output:** Cleaned CSV files written to `data/processed/`.

In [1]:
import os, sys, numpy as np, pandas as pd, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)
PROJECT_ROOT = Path.cwd().parent.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Raw data dir:', RAW_DIR)
print('Processed dir:', PROCESSED_DIR)

Project root: C:\Users\DELL\ML-project\algerian-export-ml
Raw data dir: C:\Users\DELL\ML-project\algerian-export-ml\data\raw
Processed dir: C:\Users\DELL\ML-project\algerian-export-ml\data\processed


## 1.2 – Load & Clean World Bank Data
Indicators (wide format with years as columns):
- GDP (current US$) – `NY.GDP.MKTP.CD`
- GDP growth (annual %) – `NY.GDP.MKTP.KD.ZG`
- GDP per capita (current US$) – `NY.GDP.PCAP.CD`
- Population, total – `SP.POP.TOTL`
- Merchandise trade (% of GDP) – `TG.VAL.TOTL.GD.ZS`
- Imports of goods and services (% of GDP) – `NE.IMP.GNFS.ZS`

In [2]:
wb_indicators = {
    'gdp-current-usd': ('GDP (current US$)', 'NY.GDP.MKTP.CD'),
    'gdp-growth-rate': ('GDP growth (annual %)', 'NY.GDP.MKTP.KD.ZG'),
    'gdp-per-capita': ('GDP per capita (current US$)', 'NY.GDP.PCAP.CD'),
    'population': ('Population, total', 'SP.POP.TOTL'),
    'trade-percentage-of-gdp': ('Merchandise trade (% of GDP)', 'TG.VAL.TOTL.GD.ZS'),
    'import-of-gdp': ('Imports of goods and services (% of GDP)', 'NE.IMP.GNFS.ZS'),
}
wb_base = RAW_DIR / 'worldbank'
def load_worldbank_indicator(folder, indicator_name, indicator_code):
    folder_path = wb_base / folder
    candidates = list(folder_path.glob('API_*.csv'))
    if not candidates:
        raise FileNotFoundError(f'No API CSV found in {folder_path}')
    filepath = candidates[0]
    df = pd.read_csv(filepath, skiprows=4, low_memory=False)
    df.columns = df.columns.str.strip()
    if 'Indicator Name' in df.columns:
        df = df[df['Indicator Name'] == indicator_name]
    id_vars = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
    id_vars = [c for c in id_vars if c in df.columns]
    year_cols = [c for c in df.columns if c not in id_vars]
    df_long = df.melt(id_vars=id_vars, value_vars=year_cols, var_name='year', value_name='value')
    df_long['year'] = pd.to_numeric(df_long['year'], errors='coerce')
    df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')
    df_long = df_long.dropna(subset=['year', 'value']).reset_index(drop=True)
    df_long.rename(columns={'Country Name': 'country', 'Country Code': 'country_code'}, inplace=True)
    df_long['indicator'] = indicator_code
    df_long['indicator_name'] = indicator_name
    return df_long[['country', 'country_code', 'year', 'indicator', 'indicator_name', 'value']]
wb_frames = []
for folder, (ind_name, ind_code) in wb_indicators.items():
    try:
        df_wb = load_worldbank_indicator(folder, ind_name, ind_code)
        wb_frames.append(df_wb)
        print(f'Loaded {folder}: {df_wb.shape[0]} rows')
    except Exception as e:
        print(f'ERROR loading {folder}: {e}')
df_wb_long = pd.concat(wb_frames, ignore_index=True) if wb_frames else pd.DataFrame()
print('\nCombined World Bank long-format shape:', df_wb_long.shape)
display(df_wb_long.head(10))

Loaded gdp-current-usd: 14561 rows
Loaded gdp-growth-rate: 14133 rows
Loaded gdp-per-capita: 14561 rows


Loaded population: 17195 rows
Loaded trade-percentage-of-gdp: 13900 rows
Loaded import-of-gdp: 11167 rows

Combined World Bank long-format shape: (85517, 6)


,country,country_code,year,indicator,indicator_name,value
0,Africa Eastern and Southern,AFE,"1,960.00",NY.GDP.MKTP.CD,GDP (current US$),"24,205,688,712.38"
1,Africa Western and Central,AFW,"1,960.00",NY.GDP.MKTP.CD,GDP (current US$),"11,904,805,741.68"
2,Argentina,ARG,"1,960.00",NY.GDP.MKTP.CD,GDP (current US$),"15,865,474,315.41"
3,Australia,AUS,"1,960.00",NY.GDP.MKTP.CD,GDP (current US$),"18,635,682,981.71"
4,Austria,AUT,"1,960.00",NY.GDP.MKTP.CD,GDP (current US$),"6,624,086,313.14"
5,Burundi,BDI,"1,960.00",NY.GDP.MKTP.CD,GDP (current US$),"195,999,990.00"
6,Belgium,BEL,"1,960.00",NY.GDP.MKTP.CD,GDP (current US$),"11,810,619,368.39"
7,Benin,BEN,"1,960.00",NY.GDP.MKTP.CD,GDP (current US$),"226,195,578.43"
8,Burkina Faso,BFA,"1,960.00",NY.GDP.MKTP.CD,GDP (current US$),"330,442,815.82"
9,Bangladesh,BGD,"1,960.00",NY.GDP.MKTP.CD,GDP (current US$),"4,274,894,083.33"


In [3]:
exclude_keywords = ['income', 'only', 'dividend', 'union', 'world', 'africa', 'asia', 'europe', 'america', 'arab', 'caribbean', 'pacific', 'small states', 'fragile', 'oecd', 'euro area', 'european union']
mask = ~df_wb_long['country'].str.lower().str.contains('|'.join(exclude_keywords), na=False)
df_wb_long_clean = df_wb_long[mask].copy().reset_index(drop=True)
df_wb_long_clean = df_wb_long_clean[df_wb_long_clean['year'] >= 2000].reset_index(drop=True)
print('After filtering aggregate regions & years>=2000:', df_wb_long_clean.shape)
print('Year range:', int(df_wb_long_clean['year'].min()), '-', int(df_wb_long_clean['year'].max()))
print('Unique countries:', df_wb_long_clean['country'].nunique())
print('Unique indicators:', df_wb_long_clean['indicator'].unique())
display(df_wb_long_clean.head())

After filtering aggregate regions & years>=2000: (30157, 6)
Year range: 2000 - 2024
Unique countries: 215
Unique indicators: ['NY.GDP.MKTP.CD' 'NY.GDP.MKTP.KD.ZG' 'NY.GDP.PCAP.CD' 'SP.POP.TOTL'
 'TG.VAL.TOTL.GD.ZS' 'NE.IMP.GNFS.ZS']


,country,country_code,year,indicator,indicator_name,value
0,Aruba,ABW,"2,000.00",NY.GDP.MKTP.CD,GDP (current US$),"1,873,452,513.97"
1,Afghanistan,AFG,"2,000.00",NY.GDP.MKTP.CD,GDP (current US$),"3,521,418,059.92"
2,Angola,AGO,"2,000.00",NY.GDP.MKTP.CD,GDP (current US$),"9,129,594,970.15"
3,Albania,ALB,"2,000.00",NY.GDP.MKTP.CD,GDP (current US$),"3,584,570,164.85"
4,Andorra,AND,"2,000.00",NY.GDP.MKTP.CD,GDP (current US$),"1,432,606,296.27"


In [4]:
df_wb_wide = df_wb_long_clean.pivot_table(
    index=['country', 'country_code', 'year'],
    columns='indicator',
    values='value',
    aggfunc='first'
).reset_index()
df_wb_wide.columns.name = None
df_wb_wide.columns = [str(c) for c in df_wb_wide.columns]
print('World Bank wide shape:', df_wb_wide.shape)
display(df_wb_wide.head())
wb_out = PROCESSED_DIR / '01_worldbank_cleaned.csv'
df_wb_wide.to_csv(wb_out, index=False)
print('Saved:', wb_out)

World Bank wide shape: (5375, 9)


,country,country_code,year,NE.IMP.GNFS.ZS,NY.GDP.MKTP.CD,NY.GDP.MKTP.KD.ZG,NY.GDP.PCAP.CD,SP.POP.TOTL,TG.VAL.TOTL.GD.ZS
0,Afghanistan,AFG,"2,000.00",NaN,"3,521,418,059.92",NaN,174.93,"20,130,327.00",37.29
1,Afghanistan,AFG,"2,001.00",NaN,"2,813,571,753.87",-9.43,138.71,"20,284,307.00",62.70
2,Afghanistan,AFG,"2,002.00",NaN,"3,825,701,439.00",28.60,178.95,"21,378,117.00",66.71
3,Afghanistan,AFG,"2,003.00",NaN,"4,520,946,818.55",8.83,198.87,"22,733,049.00",49.66
4,Afghanistan,AFG,"2,004.00",NaN,"5,224,896,718.68",1.41,221.76,"23,560,654.00",47.50


Saved: C:\Users\DELL\ML-project\algerian-export-ml\data\processed\01_worldbank_cleaned.csv


## 1.3 – Load & Clean WTO Data
File: `WtoData_20260325191317.csv`
Content: Merchandise trade values by product group (SITC Rev. 3 aggregates) for Algeria.

In [5]:
wto_path = RAW_DIR / 'wto_algeria' / 'WtoData_20260325191317.csv'
df_wto = pd.read_csv(wto_path, encoding='utf-8', encoding_errors='replace', low_memory=False)
df_wto.columns = df_wto.columns.str.strip().str.lower().str.replace(' ', '_')
print('WTO shape:', df_wto.shape)
print('Columns:', df_wto.columns.tolist())
display(df_wto.head())

WTO shape: (450, 24)
Columns: ['indicator_category', 'indicator_code', 'indicator', 'reporting_economy_code', 'reporting_economy_iso3a_code', 'reporting_economy', 'partner_economy_code', 'partner_economy_iso3a_code', 'partner_economy', 'product/sector_classification_code', 'product/sector_classification', 'product/sector_code', 'product/sector', 'period_code', 'period', 'frequency_code', 'frequency', 'unit_code', 'unit', 'year', 'value_flag_code', 'value_flag', 'text_value', 'value']


,indicator_category,indicator_code,indicator,reporting_economy_code,reporting_economy_iso3a_code,reporting_economy,partner_economy_code,partner_economy_iso3a_code,partner_economy,product/sector_classification_code,product/sector_classification,product/sector_code,product/sector,period_code,period,frequency_code,frequency,unit_code,unit,year,value_flag_code,value_flag,text_value,value
0,Merchandise trade values,ITS_MTV_AM,Merchandise imports by product group � annual,12,DZA,Algeria,0,NaN,World,SITC3,Merchandise - SITC Revision 3 (aggregates),AG,Agricultural products,A,Annual,A,Annual,USM,Million US dollar,2015,NaN,NaN,NaN,"10,760.83"
1,Merchandise trade values,ITS_MTV_AM,Merchandise imports by product group � annual,12,DZA,Algeria,0,NaN,World,SITC3,Merchandise - SITC Revision 3 (aggregates),AG,Agricultural products,A,Annual,A,Annual,USM,Million US dollar,2016,NaN,NaN,NaN,"9,630.08"
2,Merchandise trade values,ITS_MTV_AM,Merchandise imports by product group � annual,12,DZA,Algeria,0,NaN,World,SITC3,Merchandise - SITC Revision 3 (aggregates),AG,Agricultural products,A,Annual,A,Annual,USM,Million US dollar,2017,NaN,NaN,NaN,"9,806.63"
3,Merchandise trade values,ITS_MTV_AM,Merchandise imports by product group � annual,12,DZA,Algeria,0,NaN,World,SITC3,Merchandise - SITC Revision 3 (aggregates),AG,Agricultural products,A,Annual,A,Annual,USM,Million US dollar,2018,E,Estimate,NaN,"8,395.84"
4,Merchandise trade values,ITS_MTV_AM,Merchandise imports by product group � annual,12,DZA,Algeria,0,NaN,World,SITC3,Merchandise - SITC Revision 3 (aggregates),AG,Agricultural products,A,Annual,A,Annual,USM,Million US dollar,2019,E,Estimate,NaN,"8,840.52"


In [6]:
def clean_wto(df):
    useful = [
        'reporting_economy', 'reporting_economy_iso3a_code',
        'partner_economy', 'product/sector', 'product/sector_code',
        'indicator', 'year', 'value', 'unit'
    ]
    available = [c for c in useful if c in df.columns]
    df = df[available].copy()
    rename_map = {
        'reporting_economy': 'country',
        'reporting_economy_iso3a_code': 'country_code',
        'partner_economy': 'partner',
        'product/sector': 'product_sector',
        'product/sector_code': 'product_sector_code',
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip().str.title()
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df = df.dropna(subset=['value']).reset_index(drop=True)
    df['source'] = 'wto'
    return df
df_wto_clean = clean_wto(df_wto)
print('Cleaned WTO shape:', df_wto_clean.shape)
print('Year range:', int(df_wto_clean['year'].min()), '-', int(df_wto_clean['year'].max()))
print('Unique product sectors:', df_wto_clean['product_sector'].nunique() if 'product_sector' in df_wto_clean.columns else 'N/A')
display(df_wto_clean.head(10))

Cleaned WTO shape: (450, 10)
Year range: 2015 - 2025
Unique product sectors: 18


,country,country_code,partner,product_sector,product_sector_code,indicator,year,value,unit,source
0,Algeria,Dza,World,Agricultural Products,Ag,Merchandise Imports By Product Group � Annual,2015,"10,760.83",Million Us Dollar,wto
1,Algeria,Dza,World,Agricultural Products,Ag,Merchandise Imports By Product Group � Annual,2016,"9,630.08",Million Us Dollar,wto
2,Algeria,Dza,World,Agricultural Products,Ag,Merchandise Imports By Product Group � Annual,2017,"9,806.63",Million Us Dollar,wto
3,Algeria,Dza,World,Agricultural Products,Ag,Merchandise Imports By Product Group � Annual,2018,"8,395.84",Million Us Dollar,wto
4,Algeria,Dza,World,Agricultural Products,Ag,Merchandise Imports By Product Group � Annual,2019,"8,840.52",Million Us Dollar,wto
5,Algeria,Dza,World,Agricultural Products,Ag,Merchandise Imports By Product Group � Annual,2020,"8,943.67",Million Us Dollar,wto
6,Algeria,Dza,World,Agricultural Products,Ag,Merchandise Imports By Product Group � Annual,2021,"10,292.01",Million Us Dollar,wto
7,Algeria,Dza,World,Agricultural Products,Ag,Merchandise Imports By Product Group � Annual,2022,"12,437.89",Million Us Dollar,wto
8,Algeria,Dza,World,Agricultural Products,Ag,Merchandise Imports By Product Group � Annual,2023,"11,458.64",Million Us Dollar,wto
9,Algeria,Dza,World,Agricultural Products,Ag,Merchandise Imports By Product Group � Annual,2024,"11,623.64",Million Us Dollar,wto


In [7]:
if 'indicator' in df_wto_clean.columns:
    print(df_wto_clean['indicator'].value_counts())
wto_out = PROCESSED_DIR / '01_wto_cleaned.csv'
df_wto_clean.to_csv(wto_out, index=False)
print('Saved:', wto_out)

indicator
Merchandise Imports By Product Group � Annual    181
Merchandise Exports By Product Group � Annual    181
Total Merchandise Imports - Quarterly             44
Total Merchandise Exports - Quarterly             44
Name: count, dtype: int64
Saved: C:\Users\DELL\ML-project\algerian-export-ml\data\processed\01_wto_cleaned.csv


## 1.4 – Missing-Value Audit

In [8]:
def missing_summary(df, name):
    print(f'\n=== {name} ===')
    missing = df.isnull().mean().sort_values(ascending=False) * 100
    missing = missing[missing > 0]
    if missing.empty:
        print('No missing values.')
    else:
        print(missing.round(2).to_string())
missing_summary(df_wb_wide, 'World Bank')
missing_summary(df_wto_clean, 'WTO')


=== World Bank ===
NE.IMP.GNFS.ZS      18.70
TG.VAL.TOTL.GD.ZS    8.52
NY.GDP.MKTP.KD.ZG    4.80
NY.GDP.PCAP.CD       3.46
NY.GDP.MKTP.CD       3.46

=== WTO ===
No missing values.


## 1.5 – Next Steps
Cleaned datasets persisted in `data/processed/`:
- `01_worldbank_cleaned.csv`
- `01_wto_cleaned.csv`
Proceed to notebook `02_feature_engineering.ipynb` for metric construction.